# Lost in the Middle, revisited: free-tier reproduction

Reproduces the position experiment of *Lost in the Middle* (Liu et al., TACL 2024) on
Qwen2.5-3B-Instruct and Qwen3-4B-Instruct-2507, one model per T4. Plan and pre-registered
analysis: see the repository README.

**Kaggle settings:** Accelerator **GPU T4 x2** (not P100), Internet **on**.

**How to run:** set `STAGE` below, then *Save Version → Save & Run All (Commit)*.
Stages, in order: `smoke` → `pilot` → `qa` → `kv` → `rerun`. To resume or continue from an
earlier version, attach that version's output as an input (*Add Input → Your Work*); its
`results/` files are copied in and finished prompts are skipped.

**Map-reduce fix (phase A):** only after the baseline shows a position effect (spec §2; `qa30` is
the second setting to try). Set `MR_TASK` and `MR_SAMPLE`, then run `mr_map` → `mr_check` →
`mr_judge` → `mr_control` → `mr_reduce` → `mr_analyze`. Pilot first (`MR_SAMPLE = "pilot"`),
freeze the thresholds, then repeat with `"all"`.


In [ ]:
STAGE = "smoke"   # smoke | pilot | qa | qa30 | kv | rerun | analyze | mr_map | mr_check | mr_judge | mr_control | mr_reduce | mr_analyze
KV_LIMIT = 200    # key-value examples per position: 200 first; 500 if >5% of answers flip (plan §3)
MAX_MODEL_LEN = 20608  # longest measured prompt 20,458 tokens + 100 new tokens
ENFORCE_EAGER = False  # set True if CUDA-graph capture fails on the T4
MR_TASK = "qa20"       # map-reduce task: qa20 | qa30 | kv300, the first with a baseline position effect (spec §2)
MR_SAMPLE = "pilot"    # "pilot" (200 seeded questions) first; "all" once thresholds are frozen (spec §8)
MR_LIMIT = None        # smoke: e.g. 5 keeps the first 5 questions of the sample (runbook step 2); None otherwise
MR_CONTROL_SUBSET = None  # quota fallback (spec §7): e.g. 500 runs the control on a seeded subset; None = every question


In [ ]:
# Pre-registered guesses (plan §5): write these BEFORE the first qa/kv run and do not edit afterwards.
GUESSES = {
    "qwen2.5-3b":    {"qa20": "?", "qa30": "?", "kv300": "?"},   # U shape | Primacy only | Recency only | Flat
    "qwen3-4b-2507": {"qa20": "?", "qa30": "?", "kv300": "?"},
}
# Map-reduce (spec §6), also before its first run: Works | Flatter but worse | No effect
MR_GUESSES = {"qwen2.5-3b": "?", "qwen3-4b-2507": "?"}


## Setup: environment, authors' data, resume

In [ ]:
import os; os.chdir('/kaggle/working')

## Pipeline code (generated from the repo's `litm/` package; edit there, not here)

In [ ]:
import os
for d in ['litm', 'litm/mapreduce', 'litm/mapreduce/templates', 'litm/vendor', 'litm/vendor/lost_in_the_middle', 'litm/vendor/lost_in_the_middle/prompts']:
    os.makedirs(d, exist_ok=True)
for f in ['litm/__init__.py', 'litm/mapreduce/__init__.py', 'litm/vendor/__init__.py', 'litm/vendor/lost_in_the_middle/__init__.py']:
    open(f, "w").close()

In [ ]:
%%writefile litm/analyze.py
"""Score saved outputs and produce the tables, verdicts and figures. Runs on CPU.

    python -m litm.analyze results/ --out analysis/
"""
import argparse
import json
import os
from collections import defaultdict
from pathlib import Path

from litm.data import KV_MIDDLE, KV_POSITIONS, QA30_MIDDLE, QA30_POSITIONS, QA_MIDDLE, QA_POSITIONS
from litm.scoring import score
from litm.stats import accuracy_ci, verdict

# Paper reference curves, 20 documents, positions 1/5/10/15/20.
PAPER_QA20 = {
    "gpt-3.5-turbo-0613 (paper)": [75.8, 57.2, 53.8, 55.4, 63.2],
    "llama-2-70b-chat (authors' repo)": [56.8, 53.3, 54.1, 59.7, 69.5],
}
CURVES = {"qa20": (QA_POSITIONS, QA_MIDDLE), "qa30": (QA30_POSITIONS, QA30_MIDDLE), "kv300": (KV_POSITIONS, KV_MIDDLE)}
TITLES = {"qa20": "20 documents", "qa30": "30 documents", "kv300": "300 key-value pairs"}


def load_scores(results_dir: Path, tag: str = "") -> dict:
    """{model: {(task, position): {idx: 0/1}}} from every <model>[__tag].jsonl file with this tag."""
    scores = defaultdict(lambda: defaultdict(dict))
    for path in sorted(results_dir.glob("*.jsonl")):
        if path.name.endswith(".prompt_samples.jsonl"):
            continue
        file_tag = path.stem.split("__")[1] if "__" in path.stem else ""
        if file_tag != tag:
            continue
        with open(path, encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                scores[record["model"]][(record["task"], record["position"])][record["idx"]] = score(record)
    return scores


def summarize(scores: dict) -> dict:
    summary = {}
    for model, groups in scores.items():
        model_summary = {"accuracy": {}, "verdicts": {}}
        for (task, position), by_idx in sorted(groups.items(), key=lambda kv: (kv[0][0], kv[0][1] or 0)):
            name = task if position is None else f"{task}@{position}"
            model_summary["accuracy"][name] = accuracy_ci(list(by_idx.values()))
        for task, (positions, middle) in CURVES.items():
            if not all((task, p) in groups for p in positions):
                continue
            # Only questions present at every position, so contrasts stay paired.
            common = sorted(set.intersection(*(set(groups[(task, p)]) for p in positions)))
            first, mid, last = ([groups[(task, p)][i] for i in common] for p in (positions[0], middle, positions[-1]))
            result = verdict(first, mid, last)
            result["n"] = len(common)
            model_summary["verdicts"][task] = result
        summary[model] = model_summary
    return summary


def compare_rerun(main: dict, rerun: dict, tolerance: float = 0.02, min_agreement: float = 0.97) -> str:
    """Plan section 8: per position, rerun accuracy within +/-2 points and >=97% identical scores."""
    lines = ["| Model | Setting | Main | Rerun | Agreement | Pass |", "|---|---|---|---|---|---|"]
    all_pass = True
    for model, groups in rerun.items():
        for (task, position), by_idx in sorted(groups.items(), key=lambda kv: (kv[0][0], kv[0][1] or 0)):
            reference = main.get(model, {}).get((task, position), {})
            common = sorted(set(by_idx) & set(reference))
            if not common:
                continue
            a = sum(reference[i] for i in common) / len(common)
            b = sum(by_idx[i] for i in common) / len(common)
            agreement = sum(reference[i] == by_idx[i] for i in common) / len(common)
            ok = abs(a - b) <= tolerance and agreement >= min_agreement
            all_pass &= ok
            lines.append(f"| {model} | {task}@{position} (n={len(common)}) | {100 * a:.1f} | {100 * b:.1f} | "
                         f"{100 * agreement:.1f}% | {'yes' if ok else 'NO'} |")
    lines.append(f"\nRerun check: {'PASS' if all_pass else 'FAIL'}")
    return "\n".join(lines)


def to_markdown(summary: dict) -> str:
    lines = []
    for model, s in summary.items():
        lines += [f"## {model}", "", "| Setting | Accuracy | 95% CI | n |", "|---|---|---|---|"]
        for name, a in s["accuracy"].items():
            lines.append(f"| {name} | {100 * a['acc']:.1f} | {100 * a['low']:.1f}–{100 * a['high']:.1f} | {a['n']} |")
        for task, v in s["verdicts"].items():
            lines += ["", f"**{task} verdict: {v['shape']}** (n = {v['n']})", ""]
            for label in ("first_vs_middle", "last_vs_middle"):
                c = v[label]
                lines.append(
                    f"- {label.replace('_', ' ')}: {100 * c['diff']:+.1f} pts "
                    f"(95% CI {100 * c['low']:+.1f} to {100 * c['high']:+.1f}), McNemar p = {c['p']:.3g}, "
                    f"discordance {100 * c['discordance']:.1f}%, MDE {100 * c['mde']:.1f} pts"
                )
        lines.append("")
    return "\n".join(lines)


def plot(summary: dict, out_dir: Path) -> None:
    # Kaggle's notebook kernel exports MPLBACKEND=module://matplotlib_inline.backend_inline, which this
    # environment's matplotlib rejects at import time; plots are only ever written to files.
    os.environ["MPLBACKEND"] = "Agg"
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    for task, (positions, _) in CURVES.items():
        fig, ax = plt.subplots(figsize=(6, 4))
        x = [p + 1 for p in positions]
        for model, s in summary.items():
            points = [s["accuracy"].get(f"{task}@{p}") for p in positions]
            if not all(points):
                continue
            y = [100 * a["acc"] for a in points]
            err = [[100 * (a["acc"] - a["low"]) for a in points], [100 * (a["high"] - a["acc"]) for a in points]]
            line = ax.errorbar(x, y, yerr=err, marker="o", capsize=3, label=model)
            if task.startswith("qa"):
                for baseline, style in (("closedbook", ":"), ("oracle", "--")):
                    if baseline in s["accuracy"]:
                        ax.axhline(100 * s["accuracy"][baseline]["acc"], ls=style, color=line[0].get_color(), alpha=0.6,
                                   label=f"{model} {baseline}")
        if task == "qa20":
            for name, y in PAPER_QA20.items():
                ax.plot(x, y, marker=".", color="grey", alpha=0.5, ls="-.", label=name)
        ax.set_xticks(x)
        ax.set_xlabel("Position of the answer" if task.startswith("qa") else "Position of the key")
        ax.set_ylabel("Accuracy (%)")
        ax.set_title(TITLES[task])
        ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(out_dir / f"{task}.png", dpi=150)
        plt.close(fig)


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("results", type=Path)
    parser.add_argument("--out", type=Path, default=Path("analysis"))
    parser.add_argument("--tag", default="", help="Analyse <model>__<tag>.jsonl files instead of the main runs")
    parser.add_argument("--no-plot", action="store_true")
    args = parser.parse_args(argv)

    args.out.mkdir(parents=True, exist_ok=True)
    if args.tag == "rerun":
        report = compare_rerun(load_scores(args.results), load_scores(args.results, "rerun"))
        (args.out / "rerun_check.md").write_text(report, encoding="utf-8")
        print(report)
        return
    summary = summarize(load_scores(args.results, args.tag))
    (args.out / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
    report = to_markdown(summary)
    (args.out / "summary.md").write_text(report, encoding="utf-8")
    print(report)
    if not args.no_plot:
        plot(summary, args.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile litm/data.py
"""Load the authors' data files and turn them into prompt items.

Every item is identified by (task, position, idx), where idx is the line number in the
authors' file. The files have no ID field, so `check_qa_alignment` confirms that line i
is the same question in every file.
"""
import json
from copy import deepcopy
from pathlib import Path
from typing import Dict, Iterator, List

from xopen import xopen

from litm.vendor.lost_in_the_middle.prompting import (
    Document,
    get_closedbook_qa_prompt,
    get_kv_retrieval_prompt,
    get_qa_prompt,
)

# Pinned commit of github.com/nelson-liu/lost-in-the-middle (data re-uploaded 2024-01-04).
AUTHORS_COMMIT = "29b8a6d042ce29abccee3db1a73171a107d7e6af"

QA_POSITIONS = [0, 4, 9, 14, 19]  # 0-based; paper positions 1, 5, 10, 15, 20
KV_POSITIONS = [0, 74, 149, 224, 299]  # 0-based; positions 1, 75, 150, 225, 300
QA30_POSITIONS = [0, 4, 9, 14, 19, 24, 29]  # 0-based; the authors' 30-document files
QA_MIDDLE = 9
KV_MIDDLE = 149
QA30_MIDDLE = 14
POSITIONS = {"qa20": QA_POSITIONS, "qa30": QA30_POSITIONS, "kv300": KV_POSITIONS}
MIDDLES = {"qa20": QA_MIDDLE, "qa30": QA30_MIDDLE, "kv300": KV_MIDDLE}


def read_jsonl(path) -> List[dict]:
    with xopen(str(path)) as f:
        return [json.loads(line) for line in f]


def qa_path(root: Path, gold_index: int, n_docs: int = 20) -> Path:
    return root / "qa_data" / f"{n_docs}_total_documents" / f"nq-open-{n_docs}_total_documents_gold_at_{gold_index}.jsonl.gz"


def oracle_path(root: Path) -> Path:
    return root / "qa_data" / "nq-open-oracle.jsonl.gz"


def kv_path(root: Path, n_keys: int = 300) -> Path:
    return root / "kv_retrieval_data" / f"kv-retrieval-{n_keys}_keys.jsonl.gz"


def check_qa_alignment(root: Path) -> int:
    """Raise if any 20-doc file disagrees with the oracle file on question, answers or gold slot."""
    oracle = read_jsonl(oracle_path(root))
    for gold_index in QA_POSITIONS:
        examples = read_jsonl(qa_path(root, gold_index))
        if len(examples) != len(oracle):
            raise ValueError(f"gold_at_{gold_index}: {len(examples)} examples, oracle has {len(oracle)}")
        for idx, (example, reference) in enumerate(zip(examples, oracle)):
            if example["question"] != reference["question"] or example["answers"] != reference["answers"]:
                raise ValueError(f"gold_at_{gold_index} line {idx} does not match the oracle file")
            gold_slots = [i for i, ctx in enumerate(example["ctxs"]) if ctx.get("isgold")]
            if gold_slots != [gold_index]:
                raise ValueError(f"gold_at_{gold_index} line {idx}: gold document at {gold_slots}")
    return len(oracle)


def qa_items(root: Path, task: str, position=None, query_aware: bool = False) -> Iterator[Dict]:
    """task is 'closedbook', 'oracle', 'qa20' or 'qa30'; position is the 0-based gold index for qa20/qa30."""
    path = qa_path(root, position, int(task[2:])) if task in ("qa20", "qa30") else oracle_path(root)
    for idx, example in enumerate(read_jsonl(path)):
        if task == "closedbook":
            prompt = get_closedbook_qa_prompt(example["question"])
        else:
            documents = [Document.from_dict(ctx) for ctx in example["ctxs"]]
            prompt = get_qa_prompt(
                example["question"],
                documents,
                mention_random_ordering=False,
                query_aware_contextualization=query_aware,
            )
        yield {
            "task": task,
            "position": position,
            "idx": idx,
            "prompt": prompt,
            "answers": example["answers"],
        }


def kv_items(root: Path, position: int, n_keys: int = 300, limit=None, query_aware: bool = False) -> Iterator[Dict]:
    """Move the gold pair to `position`, as the authors' get_kv_responses_* scripts do."""
    for idx, example in enumerate(read_jsonl(kv_path(root, n_keys))):
        if limit is not None and idx >= limit:
            break
        records = deepcopy(example["ordered_kv_records"])
        gold = records.pop(records.index([example["key"], example["value"]]))
        records.insert(position, gold)
        yield {
            "task": f"kv{n_keys}",
            "position": position,
            "idx": idx,
            "prompt": get_kv_retrieval_prompt(records, example["key"], query_aware_contextualization=query_aware),
            "answers": [example["value"]],
        }


def all_items(root: Path, tasks: List[str], kv_limit=None) -> List[Dict]:
    """Build every item for the named tasks: closedbook, oracle, qa20, qa30, kv300."""
    items = []
    for task in tasks:
        if task in ("closedbook", "oracle"):
            items.extend(qa_items(root, task))
        elif task in ("qa20", "qa30"):
            for position in POSITIONS[task]:
                items.extend(qa_items(root, task, position))
        elif task == "kv300":
            for position in KV_POSITIONS:
                items.extend(kv_items(root, position, 300, limit=kv_limit))
        else:
            raise ValueError(f"Unknown task: {task}")
    return items


In [ ]:
%%writefile litm/generate.py
"""One interface for text generation: generate(prompts, params) -> [Generation].

VLLMGenerator runs on a Kaggle T4 now; phase C can add an API-backed class with the same method.
Decoding is always greedy with explicit SamplingParams: Qwen's generation_config.json would
otherwise make vLLM sample.
"""
from dataclasses import dataclass
from typing import List, Tuple

from litm.run import MODELS, render


@dataclass(frozen=True)
class GenParams:
    max_tokens: int
    logprobs: int = 1  # top-k logprobs per generated token; the chosen token is always included


@dataclass(frozen=True)
class Generation:
    text: str
    token_logprobs: List[float]           # logprob of each generated token
    token_texts: List[str]                # decoded text of each generated token
    first_top: List[Tuple[str, float]]    # top-k (token text, logprob) at the first position
    finish_reason: str                    # "stop" or "length"


def to_generation(completion) -> Generation:
    """Plain lists from a vLLM CompletionOutput (logprobs: one {token_id: Logprob} dict per token)."""
    positions = completion.logprobs or []
    chosen = [position[token_id] for token_id, position in zip(completion.token_ids, positions)]
    first_top = [(lp.decoded_token or "", lp.logprob) for lp in positions[0].values()] if positions else []
    return Generation(
        text=completion.text,
        token_logprobs=[lp.logprob for lp in chosen],
        token_texts=[lp.decoded_token or "" for lp in chosen],
        first_top=first_top,
        finish_reason=completion.finish_reason,
    )


class VLLMGenerator:
    def __init__(self, model_key: str, max_model_len: int, enforce_eager: bool = False, require_t4: bool = True):
        import torch
        from transformers import AutoTokenizer
        from vllm import LLM

        capability = torch.cuda.get_device_capability()
        if require_t4 and tuple(capability) != (7, 5):
            raise SystemExit(f"Expected a T4 (sm_75), got compute capability {capability}")
        model = MODELS[model_key]
        self.max_model_len = max_model_len
        self.tokenizer = AutoTokenizer.from_pretrained(model["hf_id"], revision=model["revision"])
        self.llm = LLM(
            model=model["hf_id"],
            revision=model["revision"],
            dtype="float16",  # T4 has no bfloat16
            attention_backend="TRITON_ATTN",
            max_model_len=max_model_len,
            gpu_memory_utilization=0.9,
            enforce_eager=enforce_eager,
            seed=0,
        )

    def generate(self, prompts: List[str], params: GenParams) -> List[Generation]:
        from vllm import SamplingParams
        from vllm.inputs import TokensPrompt

        rendered = [render(self.tokenizer, p) for p in prompts]
        ids = self.tokenizer(rendered, add_special_tokens=False)["input_ids"]
        too_long = [len(x) for x in ids if len(x) + params.max_tokens > self.max_model_len]
        if too_long:
            raise SystemExit(f"{len(too_long)} prompts exceed max_model_len {self.max_model_len} "
                             f"(longest {max(too_long)} tokens)")
        # Never pass prompt_logprobs: it disables vLLM's prefix cache.
        sampling = SamplingParams(temperature=0.0, max_tokens=params.max_tokens, logprobs=params.logprobs)
        results = self.llm.generate([TokensPrompt(prompt_token_ids=x) for x in ids], sampling, use_tqdm=False)
        return [to_generation(r.outputs[0]) for r in results]


In [ ]:
%%writefile litm/mapreduce/analyze.py
"""Score map-reduce against the baseline (spec §6, §7). Runs on CPU.

    python -m litm.mapreduce.analyze results --task qa20 --sample pilot --out analysis
"""
import argparse
import json
import math
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

from litm.analyze import load_scores
from litm.data import MIDDLES, POSITIONS
from litm.scoring import strict_em
from litm.stats import accuracy_ci, auroc, delta_u, holm, non_inferiority, range_minus_null, recovered_share, verdict

PRIMARY_ALPHA = 0.025  # one-sided, the same as a two-sided 95% CI
METHODS = ["mr", "mr_nofallback", "mr_vote", "mr_judge", "control", "gold_group", "oracle_reduce"]
GATE_OPEN = ("U shape", "Primacy only", "Recency only")


def read_rows(results: Path, pattern: str) -> list:
    rows = []
    for path in sorted(results.glob(pattern)):
        with open(path, encoding="utf-8") as f:
            rows += [json.loads(line) for line in f if line.strip()]
    return rows


def matrix(by_key: dict, positions, ids) -> np.ndarray:
    return np.array([[by_key[(p, i)]["correct"] for p in positions] for i in ids], dtype=float)


def by_field(rows, key) -> dict:
    groups = defaultdict(list)
    for r in rows:
        k = key(r)
        if k is not None:
            groups[k].append(r["correct"])
    return {str(k): {"acc": float(np.mean(v)), "n": len(v)} for k, v in sorted(groups.items())}


def mate_bin(row):
    """Mean retrieval rank of the gold's group-mates, in bins of 4 (0 = hardest distractors)."""
    return int(np.mean(row["mate_ranks"])) // 4 * 4 if row.get("mate_ranks") else None


def diagnostics(rows) -> dict:
    if not rows:
        return {}
    gold = [r for r in rows if r["is_gold_group"]]
    free = [r for r in rows if not r["is_gold_group"]]

    def share(subset, condition):
        return float(np.mean([bool(condition(r)) for r in subset])) if subset else float("nan")

    def signal(field):
        scored = [r for r in rows if r["status"] == "answer" and not r["hedged"] and r[field] is not None]
        return auroc([r[field] for r in scored], [r["cand_correct"] for r in scored])

    return {
        "not_found_gold_free": share(free, lambda r: r["status"] == "not_found"),
        "not_found_gold": share(gold, lambda r: r["status"] == "not_found"),
        "verified_gold_free": share(free, lambda r: r["verified"]),
        "verifier_false_rejection": share([r for r in gold if r["status"] == "answer" and r["cand_correct"]],
                                          lambda r: not r["verified"]),
        "invalid": share(rows, lambda r: r["status"] == "invalid"),
        "hedged": share(rows, lambda r: r["hedged"]),
        "truncated": share(rows, lambda r: r["finish_reason"] == "length"),
        "retried": share(rows, lambda r: r["attempt"] > 1),
        "p_yes_not_found": share([r for r in rows if r.get("p_yes_found") is not None],
                                 lambda r: not r["p_yes_found"]),
        "auroc_p_yes": signal("p_yes"),
        "auroc_logprob": signal("logprob"),
    }


def secondary(results: Path, model: str, task: str, methods: dict, positions, ids) -> dict:
    """Spec §5: strict exact match and answer length per method. Baseline answers are the
    first output line, as the authors score them; longer answers get free substring credit."""
    wanted = {(p, i) for p in positions for i in ids}
    out = {}
    base_rows = [r for r in read_rows(results, f"{model}.jsonl")
                 if r["task"] == task and (r["position"], r["idx"]) in wanted]
    if base_rows:
        first_lines = [r["output"].strip().split("\n")[0] for r in base_rows]
        out["baseline"] = {"strict": float(np.mean([strict_em(t, False, r["answers"]) for t, r in zip(first_lines, base_rows)])),
                           "answer_words": float(np.mean([len(t.split()) for t in first_lines]))}
    for method in METHODS:
        rows = [methods[method][k] for k in wanted if k in methods.get(method, {})]
        if rows and method != "oracle_reduce":
            out[method] = {"strict": float(np.mean([r["strict"] for r in rows])),
                           "answer_words": float(np.mean([r["answer_words"] for r in rows]))}
    return out


def analyze(results: Path, task: str, sample: str) -> dict:
    positions = POSITIONS[task]
    mid = positions.index(MIDDLES[task])
    base_scores = load_scores(results)
    finals = defaultdict(lambda: defaultdict(dict))
    for r in read_rows(results, f"*__mr-{task}-final-{sample}.jsonl"):
        finals[r["model"]][r["method"]][(r["position"], r["idx"])] = r
    diags = defaultdict(list)
    for r in read_rows(results, f"*__mr-{task}-diag-{sample}.jsonl"):
        diags[r["model"]].append(r)
    if not finals:
        raise SystemExit(f"No *__mr-{task}-final-{sample}.jsonl files in {results}: run --stage reduce first")

    models = {}
    for model, methods in sorted(finals.items()):
        base = base_scores.get(model, {})
        ids = sorted({i for _, i in methods["mr"]})
        ids = [i for i in ids if all((p, i) in methods["mr"] and i in base.get((task, p), {}) for p in positions)]
        if not ids:
            raise SystemExit(f"{model}: no questions have both baseline and map-reduce results for {task}")
        B = np.array([[base[(task, p)][i] for p in positions] for i in ids], dtype=float)
        M = matrix(methods["mr"], positions, ids)
        curves = {"baseline": [accuracy_ci(B[:, j]) for j in range(len(positions))]}
        curve_n = {"baseline": len(ids)}
        for method in METHODS:
            # a method run on fewer questions (the control on a seeded subset, spec §7) gets its curve on those
            method_ids = [i for i in ids if all((p, i) in methods.get(method, {}) for p in positions)]
            if method_ids:
                X = matrix(methods[method], positions, method_ids)
                curves[method] = [accuracy_ci(X[:, j]) for j in range(len(positions))]
                curve_n[method] = len(method_ids)
        # spec §2: the gate is decided on the whole baseline, not on the questions map-reduce ran on
        gate_ids = sorted(set.intersection(*(set(base.get((task, p), {})) for p in positions)))
        G = np.array([[base[(task, p)][i] for p in positions] for i in gate_ids], dtype=float)
        shape = verdict(G[:, 0], G[:, mid], G[:, -1])["shape"]
        mr_rows = [methods["mr"][(p, i)] for p in positions for i in ids]
        models[model] = {
            "n": len(ids),
            "gate_n": len(gate_ids),
            "baseline_verdict": shape,
            "gate_open": shape in GATE_OPEN,
            "non_inferiority": non_inferiority(M, B),
            "delta_u": delta_u(B, M),
            "recovered": recovered_share(B, M, mid),
            "range": {"baseline": range_minus_null(B), "mr": range_minus_null(M)},
            "curves": curves,
            "curve_n": curve_n,
            "by_slot": by_field(mr_rows, lambda r: r["gold_slot"]),
            "by_mate_rank": by_field(mr_rows, mate_bin),
            "paths": dict(Counter(r["path"] for r in mr_rows)),
            "diagnostics": diagnostics(diags[model]),
            "secondary": secondary(results, model, task, methods, positions, ids),
            "reference": {name: accuracy_ci(list(base[(name, None)].values()))
                          for name in ("closedbook", "oracle") if (name, None) in base},
        }

    names = list(models)
    for test in ("non_inferiority", "delta_u"):
        for name, rejected in zip(names, holm([models[n][test]["p"] for n in names], alpha=PRIMARY_ALPHA)):
            models[name][test]["holm_pass"] = bool(rejected)
    for name in names:
        ni, du = models[name]["non_inferiority"]["holm_pass"], models[name]["delta_u"]["holm_pass"]
        models[name]["outcome"] = "Works" if ni and du else "Flatter but worse" if du else "No effect"
    return {"task": task, "sample": sample, "positions": [p + 1 for p in positions], "models": models}


def pct(x) -> str:
    return "n/a" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{100 * x:.1f}"


def to_markdown(summary: dict) -> str:
    lines = [f"# Map-reduce vs baseline: {summary['task']} ({summary['sample']})", ""]
    for model, s in summary["models"].items():
        ni, du, rec = s["non_inferiority"], s["delta_u"], s["recovered"]
        lines += [f"## {model}: **{s['outcome']}** (n = {s['n']} questions)", ""]
        if not s["gate_open"]:
            lines += [f"> Gate closed: the baseline verdict (all {s['gate_n']} baseline questions) is *{s['baseline_verdict']}*, so there is no "
                      "position effect for map-reduce to fix (spec section 2). Report the outcome as moot.", ""]
        lines += [
            f"- Baseline verdict (gate, all {s['gate_n']} baseline questions): {s['baseline_verdict']}",
            f"- Test 1, non-inferiority (margin {pct(ni['margin'])} pts): MR - baseline = {pct(ni['diff'])} pts "
            f"(95% CI {pct(ni['low'])} to {pct(ni['high'])}), pass after Holm: {ni['holm_pass']}",
            f"- Test 2, flatter: U baseline {pct(du['u_base'])} vs MR {pct(du['u_new'])}; dU = {pct(du['diff'])} pts "
            f"(95% CI {pct(du['low'])} to {pct(du['high'])}), pass after Holm: {du['holm_pass']}",
            f"- Test 3, middle gap recovered: {pct(rec['share'])}% (95% CI {pct(rec['low'])} to {pct(rec['high'])})"
            + (" (unstable: the baseline gap's CI touches 0)" if rec["unstable"] else ""),
            f"- Best-worst range (descriptive): baseline {pct(s['range']['baseline']['range'])} "
            f"(no-effect value {pct(s['range']['baseline']['null_mean'])}), MR {pct(s['range']['mr']['range'])} "
            f"(no-effect value {pct(s['range']['mr']['null_mean'])})",
            "", "| Method | " + " | ".join(f"pos {p}" for p in summary["positions"]) + " |",
            "|---|" + "---|" * len(summary["positions"]),
        ]
        for method, points in s["curves"].items():
            n = s["curve_n"][method]
            label = method if n == s["n"] else f"{method} (n = {n})"
            lines.append(f"| {label} | " + " | ".join(pct(a["acc"]) for a in points) + " |")
        for name, a in s["reference"].items():
            lines.append(f"\n{name}: {pct(a['acc'])}")
        lines += ["", "**Accuracy by gold slot inside its group (MR):** "
                  + ", ".join(f"slot {int(k) + 1}: {pct(v['acc'])} (n={v['n']})" for k, v in s["by_slot"].items())]
        if s["by_mate_rank"]:
            lines.append("**By mean rank of the gold's group-mates (0 = hardest):** "
                         + ", ".join(f"{k}+: {pct(v['acc'])} (n={v['n']})" for k, v in s["by_mate_rank"].items()))
        lines.append("**Strict exact match / mean answer words:** " + ", ".join(
            f"{m} {pct(v['strict'])}% / {v['answer_words']:.1f}" for m, v in s["secondary"].items()))
        lines.append("**Decision paths (MR):** " + ", ".join(f"{k} {v}" for k, v in sorted(s["paths"].items())))
        lines += ["", "**Map-stage diagnostics:**"]
        lines += [f"- {k.replace('_', ' ')}: {v:.3f}" if k.startswith("auroc") else f"- {k.replace('_', ' ')}: {pct(v)}%"
                  for k, v in s["diagnostics"].items()]
        lines.append("")
    return "\n".join(lines)


def plot(summary: dict, out_dir: Path) -> None:
    # Kaggle's notebook kernel exports MPLBACKEND=module://matplotlib_inline.backend_inline, which this
    # environment's matplotlib rejects at import time; plots are only ever written to files.
    os.environ["MPLBACKEND"] = "Agg"
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    x = summary["positions"]
    for model, s in summary["models"].items():
        fig, ax = plt.subplots(figsize=(6, 4))
        for method in ("baseline", "mr", "control"):
            if method in s["curves"]:
                points = s["curves"][method]
                y = [100 * a["acc"] for a in points]
                err = [[100 * (a["acc"] - a["low"]) for a in points], [100 * (a["high"] - a["acc"]) for a in points]]
                ax.errorbar(x, y, yerr=err, marker="o", capsize=3, label=method)
        for name, style in (("closedbook", ":"), ("oracle", "--")):
            if name in s["reference"]:
                ax.axhline(100 * s["reference"][name]["acc"], ls=style, color="grey", label=name)
        ax.set_xticks(x)
        ax.set_xlabel("Position of the answer")
        ax.set_ylabel("Accuracy (%)")
        ax.set_title(f"{model}: {summary['task']}, map-reduce vs one prompt")
        ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(out_dir / f"mr_{summary['task']}_{model}.png", dpi=150)
        plt.close(fig)


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("results", type=Path)
    parser.add_argument("--task", choices=sorted(POSITIONS), default="qa20")
    parser.add_argument("--sample", choices=["pilot", "all"], default="pilot")
    parser.add_argument("--out", type=Path, default=Path("analysis"))
    parser.add_argument("--no-plot", action="store_true")
    args = parser.parse_args(argv)

    args.out.mkdir(parents=True, exist_ok=True)
    summary = analyze(args.results, args.task, args.sample)
    stem = f"mr_{args.task}_{args.sample}"
    (args.out / f"{stem}.json").write_text(json.dumps(summary, indent=2, default=float))
    report = to_markdown(summary)
    (args.out / f"{stem}.md").write_text(report, encoding="utf-8")
    print(report)
    if not args.no_plot:
        plot(summary, args.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile litm/mapreduce/choose.py
"""Choose one final answer from the groups' candidates (spec §4.4, §4.5, §9).

No rule ever breaks a tie by group order: that would rebuild a position bias into the reduce step.
Exact ties go to the alphabetically first normalized answer.
"""
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Callable, List, Optional, Sequence

from litm.vendor.lost_in_the_middle.metrics import normalize_answer


@dataclass(frozen=True)
class Candidate:
    group: int
    answer: str
    evidence: str
    verified: bool
    hedged: bool
    p_yes: Optional[float] = None    # from the shared yes/no prompt (primary rule)
    p_yes_found: Optional[bool] = None  # False: neither Yes nor No in the top tokens, p_yes is a 0.5 stand-in
    logprob: Optional[float] = None  # mean answer-token logprob (sensitivity rule)


@dataclass(frozen=True)
class Decision:
    answer: str      # "" when abstained
    path: str        # "verified" | "fallback" | "judge" | "abstain" | ...
    abstained: bool


ABSTAIN = Decision("", "abstain", True)


def _usable(cands: Sequence[Candidate]) -> List[Candidate]:
    return [c for c in cands if not c.hedged]


def _pick(cands: Sequence[Candidate], score: Callable) -> str:
    """The normalized answer whose best supporting candidate scores highest; return that
    candidate's surface text."""
    by_answer = defaultdict(list)
    for c in cands:
        by_answer[normalize_answer(c.answer)].append(c)
    best = max(sorted(by_answer), key=lambda norm: max(score(c) for c in by_answer[norm]))
    return max(by_answer[best], key=score).answer


def _p(c: Candidate) -> float:
    if c.p_yes is None:
        raise ValueError(f"candidate {c.answer!r} from group {c.group} has no yes/no score")
    return c.p_yes


def _logprob(c: Candidate) -> float:
    return c.logprob if c.logprob is not None else float("-inf")


def choose_primary(cands: Sequence[Candidate], use_fallback: bool = True) -> Decision:
    """Highest P(Yes) among verified answers; else (fallback) among unverified ones; else abstain."""
    usable = _usable(cands)
    verified = [c for c in usable if c.verified]
    if verified:
        return Decision(_pick(verified, _p), "verified", False)
    if use_fallback and usable:
        return Decision(_pick(usable, _p), "fallback", False)
    return ABSTAIN


def choose_vote(cands: Sequence[Candidate]) -> Decision:
    """The original design: most votes among verified answers, ties by mean answer logprob."""
    usable = _usable(cands)
    verified = [c for c in usable if c.verified]
    if verified:
        votes = Counter(normalize_answer(c.answer) for c in verified)
        return Decision(_pick(verified, lambda c: (votes[normalize_answer(c.answer)], _logprob(c))), "verified", False)
    if usable:
        return Decision(_pick(usable, _logprob), "fallback", False)
    return ABSTAIN


def distinct_verified(cands: Sequence[Candidate]) -> List[str]:
    return sorted({normalize_answer(c.answer) for c in _usable(cands) if c.verified})


def choose_judge(cands: Sequence[Candidate], judged: Optional[str]) -> Decision:
    """The judge's pick (a normalized answer) when there were 2+ verified answers; else the primary rule."""
    if judged is not None and len(distinct_verified(cands)) >= 2:
        for c in _usable(cands):
            if c.verified and normalize_answer(c.answer) == judged:
                return Decision(c.answer, "judge", False)
    return choose_primary(cands)


def choose_kv(cands: Sequence[Candidate]) -> Decision:
    """Keys are unique, so a verified value can only come from the gold group."""
    verified = [c for c in cands if c.verified]
    return Decision(verified[0].answer, "verified", False) if verified else ABSTAIN


In [ ]:
%%writefile litm/mapreduce/parse.py
"""Read map, yes/no and judge outputs (spec §4.2)."""
import math
import re
from dataclasses import dataclass
from typing import Iterable, Optional, Sequence, Tuple

from litm.vendor.lost_in_the_middle.metrics import normalize_answer

HEDGE_RE = re.compile(r"\bor\b|;", re.I)
EVIDENCE_RE = re.compile(r"^[ \t*_#>-]*evidence[ \t]*[*_]*[ \t]*:[ \t]*[*_]*[ \t]*(.*)", re.I | re.M | re.S)
JUDGE_RE = re.compile(r"choice[ \t]*[*_]*[ \t]*:[ \t]*[*_\[( \t]*(\d+)", re.I)


def _field(label: str) -> "re.Pattern":
    """A line like 'Answer: X', '**Answer:** X' or 'answer : X' (any case); group 1 is X."""
    return re.compile(r"^[ \t*_#>-]*" + label + r"[ \t]*[*_]*[ \t]*:[ \t]*[*_]*[ \t]*(.*?)[ \t*_]*$", re.I | re.M)


@dataclass(frozen=True)
class Parsed:
    status: str  # "answer" | "not_found" | "invalid"
    answer: str = ""
    evidence: str = ""
    hedged: bool = False
    reason: str = ""


def broken_reason(text: str, token_logprobs: Sequence[float] = ()) -> str:
    """Signs of fp16 overflow: a NaN/inf logprob, or output that is only '!' characters."""
    if any(math.isnan(x) or math.isinf(x) for x in token_logprobs):
        return "bad logprob"
    if re.fullmatch(r"\s*!+\s*", text):
        return "only !"
    return ""


def _body_start(text: str) -> int:
    """Skip any <think>...</think> block."""
    end = text.rfind("</think>")
    return 0 if end < 0 else end + len("</think>")


def parse_map_output(text: str, token_logprobs: Sequence[float] = (), label: str = "answer") -> Parsed:
    reason = broken_reason(text, token_logprobs)
    if reason:
        return Parsed("invalid", reason=reason)
    body = text[_body_start(text):]
    match = _field(label).search(body)
    if not match:
        return Parsed("invalid", reason=f"no {label} line")
    answer = match.group(1).strip()
    if not answer:
        return Parsed("invalid", reason="empty answer")
    if normalize_answer(answer).startswith("not found"):
        return Parsed("not_found")
    evidence = EVIDENCE_RE.search(body, match.end())
    return Parsed("answer", answer, evidence.group(1).strip() if evidence else "", bool(HEDGE_RE.search(answer)))


def answer_logprob(token_texts: Sequence[str], token_logprobs: Sequence[float], label: str = "answer") -> Optional[float]:
    """Mean logprob of the tokens that overlap the answer text (sensitivity tie-break only)."""
    text = "".join(token_texts)
    match = _field(label).search(text, _body_start(text))
    if not match or match.start(1) == match.end(1):
        return None
    start, end, pos, picked = match.start(1), match.end(1), 0, []
    for piece, logprob in zip(token_texts, token_logprobs):
        if pos < end and pos + len(piece) > start:
            picked.append(logprob)
        pos += len(piece)
    return sum(picked) / len(picked) if picked else None


def p_yes(top: Iterable[Tuple[str, float]]) -> Tuple[float, bool]:
    """P(Yes) / (P(Yes) + P(No)) from the first token's top logprobs; (0.5, False) if neither shows up."""
    yes = no = 0.0
    for token, logprob in top:
        if not math.isfinite(logprob):  # a broken fp16 output; NaN would win or lose every comparison
            continue
        word = token.strip().lower()
        if word == "yes":
            yes += math.exp(logprob)
        elif word == "no":
            no += math.exp(logprob)
    if yes + no == 0:
        return 0.5, False
    return yes / (yes + no), True


def parse_judge(text: str, n_candidates: int) -> Optional[int]:
    """0-based index of the judge's choice, or None if missing or out of range."""
    match = JUDGE_RE.search(text)
    if not match:
        return None
    choice = int(match.group(1)) - 1
    return choice if 0 <= choice < n_candidates else None


In [ ]:
%%writefile litm/mapreduce/prompts.py
"""Prompt builders for map, yes/no check and judge calls (spec §4).

Document and key-value lines are formatted exactly as the authors' prompting.py does. Templates
are read in text mode, so CRLF checkouts produce the same prompts.
"""
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

TEMPLATES = Path(__file__).parent / "templates"


def _template(name: str) -> str:
    with open(TEMPLATES / name, encoding="utf-8") as f:
        return f.read().rstrip("\n")


def format_documents(docs: Sequence[Dict]) -> str:
    return "\n".join(f"Document [{i + 1}](Title: {d['title']}) {d['text']}" for i, d in enumerate(docs))


def format_kv_records(pairs: Sequence[Sequence[str]]) -> str:
    """The authors' JSON-ish serialisation, without their check that the query key is present."""
    out = ""
    for index, (key, value) in enumerate(pairs):
        start = "{" if index == 0 else " "
        end = ",\n" if index != len(pairs) - 1 else "}"
        out += f'{start}"{key}": "{value}"{end}'
    return out


def qa_map_prompt(question: str, docs: Sequence[Dict]) -> str:
    return _template("qa_map.prompt").format(question=question, search_results=format_documents(docs))


def kv_map_prompt(pairs: Sequence[Sequence[str]], key: str) -> str:
    return _template("kv_map.prompt").format(formatted_kv_records=format_kv_records(pairs), key=key)


def map_prompt(item: Dict) -> str:
    if item["task"].startswith("kv"):
        return kv_map_prompt(item["docs"], item["query_key"])
    return qa_map_prompt(item["question"], item["docs"])


def check_prompt(question: str, evidence: str, answer: str) -> str:
    return _template("qa_check.prompt").format(question=question, evidence=evidence, answer=answer)


def judge_prompt(question: str, candidates: List[Tuple[str, str]]) -> str:
    lines = "\n".join(f"[{i + 1}] Answer: {a}\n    Sentence: {e}" for i, (a, e) in enumerate(candidates))
    return _template("qa_judge.prompt").format(question=question, candidates=lines)


In [ ]:
%%writefile litm/mapreduce/run.py
"""Map-reduce runs (spec: docs/superpowers/specs/2026-09-30-map-reduce-fix-design.md).

GPU stages, one model per GPU, in this order (the last three are QA only):
    python -m litm.mapreduce.run --model qwen2.5-3b --task qa20 --stage map --data <authors repo> --out results --questions pilot
    ... --stage check / judge / control
then on CPU:
    ... --stage reduce

Every generated output is saved in <model>__mr-<stage>.jsonl, keyed by a hash of the exact prompt and
generation settings, so identical groups are generated once, reruns resume, and any prompt change
invalidates old outputs automatically. `reduce` writes <model>__mr-<task>-final-<sample>.jsonl (one
row per question, position and method) and <model>__mr-<task>-diag-<sample>.jsonl (one row per group).
"""
import argparse
import dataclasses
import json
from collections import defaultdict
from pathlib import Path

import numpy as np

from litm.data import POSITIONS, kv_path, oracle_path, qa_path, read_jsonl
from litm.generate import GenParams
from litm.mapreduce.choose import (
    ABSTAIN,
    Candidate,
    Decision,
    choose_judge,
    choose_kv,
    choose_primary,
    choose_vote,
    distinct_verified,
)
from litm.mapreduce.parse import answer_logprob, broken_reason, p_yes, parse_judge, parse_map_output
from litm.mapreduce.prompts import check_prompt, judge_prompt, map_prompt, qa_map_prompt
from litm.mapreduce.split import SEED, content_key, kv_map_items, qa_map_items, sample_ids
from litm.mapreduce.verify import verify_kv, verify_qa
from litm.run import MODELS, environment
from litm.scoring import score_final, score_final_kv, strict_em
from litm.vendor.lost_in_the_middle.metrics import normalize_answer

MAP_PARAMS = GenParams(max_tokens=150, logprobs=1)
CHECK_PARAMS = GenParams(max_tokens=1, logprobs=20)
JUDGE_PARAMS = GenParams(max_tokens=10, logprobs=1)
CONTROL_PARAMS = MAP_PARAMS
SHORT_MAX_LEN = 2048    # map, check and judge prompts are at most 1.5K tokens (6-document qa30 groups)
CONTROL_MAX_LEN = 8192  # 20 or 30 documents in one prompt (about 3.1K / 4.7K tokens)
GPU_STAGES = ("map", "check", "judge", "control")
QA_ONLY_STAGES = ("check", "judge", "control")


# ---------------------------------------------------------------- files and resumable generation

def stage_path(args, stage: str) -> Path:
    return args.out / f"{args.model}__mr-{stage}.jsonl"


def result_path(args, kind: str) -> Path:
    return args.out / f"{args.model}__mr-{args.task}-{kind}-{args.questions}.jsonl"


def job_key(prompt: str, params: GenParams) -> str:
    return content_key(prompt, params.max_tokens, params.logprobs)


def load_records(path: Path) -> dict:
    """{key: record}; a later attempt for the same key replaces an earlier one. A last line cut off
    by a killed session (no final newline, possibly mid-character) is skipped, so its prompt is
    simply generated again; a bad line elsewhere raises."""
    records = {}
    if path.exists():
        data = path.read_bytes()
        text = data[:data.rfind(b"\n") + 1].decode("utf-8")
        for line in text.splitlines():
            if not line.strip():
                continue
            record = json.loads(line)
            old = records.get(record["key"])
            if old is None or record["attempt"] >= old["attempt"]:
                records[record["key"]] = record
    return records


def cut_torn_tail(path: Path) -> None:
    """Drop a last line left half-written by a killed session, so the next append starts on a fresh line."""
    data = path.read_bytes()
    if data and not data.endswith(b"\n"):
        path.write_bytes(data[:data.rfind(b"\n") + 1])


def subset_ids(ids, n):
    """A seeded n-question subset of `ids`, sorted (spec §7: the control may run on a seeded subset)."""
    if n is None or n >= len(ids):
        return list(ids)
    rng = np.random.default_rng([SEED, n])
    return sorted(ids[i] for i in rng.choice(len(ids), size=n, replace=False))


def run_prompts(gen, path: Path, prompts, params: GenParams, batch_size: int) -> None:
    """Generate every prompt not yet saved in `path`, appending after each batch.
    A broken output (NaN logprob, only '!') is re-run once, in a later batch."""
    prompts = list(dict.fromkeys(prompts))
    path.touch()  # an empty file still records that the stage ran (reduce checks for it)
    cut_torn_tail(path)
    done = load_records(path)
    todo = []
    for prompt in prompts:
        previous = done.get(job_key(prompt, params))
        if previous is None:
            todo.append((prompt, 1))
        elif previous["attempt"] == 1 and broken_reason(previous["text"], previous["token_logprobs"]):
            todo.append((prompt, 2))
    print(f"{path.name}: {len(prompts)} prompts, {len(prompts) - len(todo)} done, {len(todo)} to run", flush=True)
    while todo:
        retry = []
        with open(path, "a", encoding="utf-8") as f:
            for start in range(0, len(todo), batch_size):
                batch = todo[start:start + batch_size]
                outputs = gen.generate([prompt for prompt, _ in batch], params)
                for (prompt, attempt), g in zip(batch, outputs):
                    record = {"key": job_key(prompt, params), "attempt": attempt, "text": g.text,
                              "token_logprobs": g.token_logprobs, "token_texts": g.token_texts,
                              "first_top": g.first_top, "finish_reason": g.finish_reason}
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    if attempt == 1 and broken_reason(g.text, g.token_logprobs):
                        retry.append((prompt, 2))
                f.flush()
                print(f"{path.name}: {min(start + batch_size, len(todo))}/{len(todo)}", flush=True)
        todo = retry


def make_generator(model: str, max_model_len: int, enforce_eager: bool, meta_path: Path):
    from litm.generate import VLLMGenerator

    gen = VLLMGenerator(model, max_model_len, enforce_eager)
    meta = {"model": model, **MODELS[model], "max_model_len": max_model_len, "environment": environment()}
    meta_path.write_text(json.dumps(meta, indent=2))
    return gen


# ---------------------------------------------------------------- items

def question_ids(args) -> list:
    if args.task.startswith("kv"):
        n_total = len(read_jsonl(kv_path(args.data)))
        if args.kv_limit is not None:
            n_total = min(n_total, args.kv_limit)
    else:
        n_total = len(read_jsonl(oracle_path(args.data)))
    return sample_ids(n_total, args.questions, args.limit)


def map_items(args) -> list:
    ids = question_ids(args)
    if args.task.startswith("kv"):
        return kv_map_items(args.data, POSITIONS[args.task], ids)
    return qa_map_items(args.data, int(args.task[2:]), POSITIONS[args.task], ids)


def question_groups(items) -> dict:
    """{(position, idx): [its group items]}"""
    groups = defaultdict(list)
    for item in items:
        groups[(item["position"], item["idx"])].append(item)
    return dict(sorted(groups.items()))


def control_prompts(args) -> dict:
    """{(position, idx): all documents in one prompt, original order, map prompt wording} (spec §7)."""
    ids, n_docs, prompts = set(subset_ids(question_ids(args), args.control_subset)), int(args.task[2:]), {}
    for position in POSITIONS[args.task]:
        for idx, example in enumerate(read_jsonl(qa_path(args.data, position, n_docs))):
            if idx in ids:
                docs = [{"title": c["title"], "text": c["text"]} for c in example["ctxs"]]
                prompts[(position, idx)] = qa_map_prompt(example["question"], docs)
    return prompts


# ---------------------------------------------------------------- parse + verify one question's groups

def collect(groups, maps: dict):
    """-> (candidates, rows) where rows are (item, map record, parsed, verdict) for every group."""
    cands, rows = [], []
    for item in groups:
        kv = item["task"].startswith("kv")
        label = "value" if kv else "answer"
        record = maps.get(job_key(map_prompt(item), MAP_PARAMS))
        if record is None:
            raise SystemExit(f"Missing map output for {item['task']} line {item['idx']} group {item['group']}: "
                             "run --stage map first")
        parsed = parse_map_output(record["text"], record["token_logprobs"], label)
        verdict = None
        if parsed.status == "answer":
            if kv:
                verdict = verify_kv(parsed.answer, parsed.evidence, item["docs"], item["query_key"])
            else:
                verdict = verify_qa(parsed.answer, parsed.evidence, item["docs"])
            cands.append(Candidate(item["group"], parsed.answer, parsed.evidence, verdict.verified, parsed.hedged,
                                   None, answer_logprob(record["token_texts"], record["token_logprobs"], label)))
        rows.append((item, record, parsed, verdict))
    return cands, rows


def lookup_p_yes(checks: dict, question: str, cand: Candidate) -> Candidate:
    """`cand` with P(Yes) filled in; p_yes_found is False when neither Yes nor No was in the top tokens (0.5 then)."""
    record = checks.get(job_key(check_prompt(question, cand.evidence, cand.answer), CHECK_PARAMS))
    if record is None:
        raise SystemExit(f"Missing yes/no output for {cand.answer!r}: run --stage check first")
    value, found = p_yes(record["first_top"])
    return dataclasses.replace(cand, p_yes=value, p_yes_found=found)


def judge_prompt_for(question: str, cands, position: int, idx: int):
    """(prompt, shown order of normalized answers), or None with fewer than two verified answers."""
    distinct = distinct_verified(cands)
    if len(distinct) < 2:
        return None
    order = [distinct[i] for i in np.random.default_rng([SEED, position, idx]).permutation(len(distinct))]
    shown = {}
    for c in cands:
        norm = normalize_answer(c.answer)
        if c.verified and not c.hedged and norm not in shown:
            shown[norm] = c
    return judge_prompt(question, [(shown[n].answer, shown[n].evidence) for n in order]), order


# ---------------------------------------------------------------- GPU stages

def stage_map(args, gen) -> None:
    prompts = [map_prompt(item) for item in map_items(args)]
    run_prompts(gen, stage_path(args, "map"), prompts, MAP_PARAMS, args.batch_size)


def stage_check(args, gen) -> None:
    maps, prompts = load_records(stage_path(args, "map")), []
    for groups in question_groups(map_items(args)).values():
        cands, _ = collect(groups, maps)
        prompts += [check_prompt(groups[0]["question"], c.evidence, c.answer) for c in cands if not c.hedged]
    run_prompts(gen, stage_path(args, "check"), prompts, CHECK_PARAMS, args.batch_size)


def stage_judge(args, gen) -> None:
    maps, prompts = load_records(stage_path(args, "map")), []
    for (position, idx), groups in question_groups(map_items(args)).items():
        cands, _ = collect(groups, maps)
        built = judge_prompt_for(groups[0]["question"], cands, position, idx)
        if built is not None:
            prompts.append(built[0])
    run_prompts(gen, stage_path(args, "judge"), prompts, JUDGE_PARAMS, args.batch_size)


def stage_control(args, gen) -> None:
    run_prompts(gen, stage_path(args, "control"), list(control_prompts(args).values()), CONTROL_PARAMS, args.batch_size)


# ---------------------------------------------------------------- reduce (CPU)

def reduce_task(args) -> None:
    kv = args.task.startswith("kv")
    maps = load_records(stage_path(args, "map"))
    checks = {} if kv else load_records(stage_path(args, "check"))
    judge_file = stage_path(args, "judge")
    judges = load_records(judge_file) if not kv and judge_file.exists() else None
    controls = {} if kv else load_records(stage_path(args, "control"))
    control = {} if kv else control_prompts(args)
    finals, diags = [], []

    for (position, idx), groups in question_groups(map_items(args)).items():
        question, answers = groups[0]["question"], groups[0]["answers"]

        def score(decision: Decision) -> int:
            if kv:
                return score_final_kv(decision.answer, decision.abstained, answers[0])
            return score_final(decision.answer, decision.abstained, answers)

        cands, rows = collect(groups, maps)
        if not kv:
            cands = [c if c.hedged else lookup_p_yes(checks, question, c) for c in cands]
        gold_item, _, gold_parsed, _ = next(row for row in rows if row[0]["is_gold_group"])
        decisions = {"gold_group": Decision(gold_parsed.answer, "gold_group", False)
                     if gold_parsed.status == "answer" else ABSTAIN}
        if kv:
            decisions["mr"] = choose_kv(cands)
        else:
            decisions["mr"] = choose_primary(cands)
            decisions["mr_nofallback"] = choose_primary(cands, use_fallback=False)
            decisions["mr_vote"] = choose_vote(cands)
            if judges is not None:
                judged, built = None, judge_prompt_for(question, cands, position, idx)
                if built is not None:
                    record = judges.get(job_key(built[0], JUDGE_PARAMS))
                    if record is None:
                        raise SystemExit(f"Missing judge output for {args.task} line {idx}: run --stage judge first")
                    choice = parse_judge(record["text"], len(built[1]))
                    judged = None if choice is None else built[1][choice]
                decisions["mr_judge"] = choose_judge(cands, judged)
            prompt = control.get((position, idx))  # absent outside a --control-subset
            record = controls.get(job_key(prompt, CONTROL_PARAMS)) if prompt else None
            if record is not None:
                parsed = parse_map_output(record["text"], record["token_logprobs"])
                decisions["control"] = Decision(parsed.answer, "control", False) if parsed.status == "answer" else ABSTAIN

        common = {"model": args.model, "task": args.task, "position": position, "idx": idx,
                  "gold_group": gold_item["group"], "gold_slot": gold_item["gold_slot"],
                  "mate_ranks": gold_item["mate_ranks"], "n_candidates": len(cands),
                  "n_verified": len(distinct_verified(cands))}
        for method, d in decisions.items():
            finals.append({**common, "method": method, "answer": d.answer, "abstained": d.abstained, "path": d.path,
                           "correct": score(d), "strict": 0 if kv else strict_em(d.answer, d.abstained, answers),
                           "answer_words": len(d.answer.split())})
        oracle = int(any(score(Decision(c.answer, "", False)) for c in cands))
        finals.append({**common, "method": "oracle_reduce", "answer": "", "abstained": False, "path": "oracle",
                       "correct": oracle, "strict": 0, "answer_words": 0})

        by_group = {c.group: c for c in cands}
        for item, record, parsed, verdict in rows:
            cand = by_group.get(item["group"])
            diags.append({
                "model": args.model, "task": args.task, "position": position, "idx": idx, "group": item["group"],
                "is_gold_group": item["is_gold_group"], "n_answer_bearing": item["n_answer_bearing"],
                "status": parsed.status, "reason": parsed.reason, "hedged": parsed.hedged,
                "verified": bool(verdict and verdict.verified), "verify_how": verdict.how if verdict else "",
                "finish_reason": record["finish_reason"], "attempt": record["attempt"],
                "cand_correct": score(Decision(parsed.answer, "", False)) if parsed.status == "answer" else 0,
                "p_yes": cand.p_yes if cand else None, "p_yes_found": cand.p_yes_found if cand else None,
                "logprob": cand.logprob if cand else None,
            })

    for kind, rows in (("final", finals), ("diag", diags)):
        with open(result_path(args, kind), "w", encoding="utf-8") as f:
            f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
    print(f"reduce: {len(finals)} final rows, {len(diags)} group rows -> {result_path(args, 'final').name}")


# ---------------------------------------------------------------- CLI

STAGES = {"map": stage_map, "check": stage_check, "judge": stage_judge, "control": stage_control}


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=sorted(MODELS), required=True)
    parser.add_argument("--task", choices=sorted(POSITIONS), required=True)
    parser.add_argument("--stage", choices=[*GPU_STAGES, "reduce"], required=True)
    parser.add_argument("--data", type=Path, required=True, help="Path to the authors' repo checkout")
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--questions", choices=["pilot", "all"], default="pilot")
    parser.add_argument("--limit", type=int, default=None, help="Keep only the first N questions of the sample")
    parser.add_argument("--kv-limit", type=int, default=None, help="Key-value: use the first N examples, as the baseline did")
    parser.add_argument("--control-subset", type=int, default=None,
                        help="Control stage: run on a seeded subset of N questions (spec §7); reduce uses whatever control outputs exist")
    parser.add_argument("--batch-size", type=int, default=256)
    parser.add_argument("--enforce-eager", action="store_true")
    args = parser.parse_args(argv)

    if args.task.startswith("kv") and args.stage in QA_ONLY_STAGES:
        raise SystemExit(f"Stage {args.stage} is not used for {args.task} (spec §9)")
    args.out.mkdir(parents=True, exist_ok=True)
    if args.stage == "reduce":
        reduce_task(args)
        return
    max_model_len = CONTROL_MAX_LEN if args.stage == "control" else SHORT_MAX_LEN
    gen = make_generator(args.model, max_model_len, args.enforce_eager,
                         args.out / f"{args.model}__mr-{args.stage}.meta.json")
    STAGES[args.stage](args, gen)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile litm/mapreduce/split.py
"""Cut each prompt's documents (or key-value pairs) into small groups for map calls (spec §3, §9).

Groups are contiguous blocks in the original order. Inside the gold's group, the gold is moved to
a slot assigned by a seeded, balanced schedule, so each slot is used equally often at every global
position. Nothing else moves.
"""
import hashlib
import json
from copy import deepcopy
from math import ceil
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np

from litm.data import kv_path, qa_path, read_jsonl
from litm.vendor.lost_in_the_middle.metrics import normalize_answer

SEED = 0
QA_GROUP_SIZES = {20: 4, 30: 6}  # even groups only: 30 as 6x4 + 2x3 would give the last positions an easier gold group
KV_GROUP_SIZE = 20
PILOT_SIZE = 200


def group_sizes(n_items: int, group_size: int) -> List[int]:
    """As even as possible, none above group_size: 20 -> 5x4, 30 -> 5x6, 300 -> 15x20."""
    n_groups = ceil(n_items / group_size)
    base, extra = divmod(n_items, n_groups)
    return [base + 1] * extra + [base] * (n_groups - extra)


def group_of(index: int, sizes: Sequence[int]) -> int:
    """Which group the item at `index` falls in."""
    return int(np.searchsorted(np.cumsum(sizes), index, side="right"))


def gold_slots(n_questions: int, position: int, group_size: int, seed: int = SEED) -> np.ndarray:
    """Slot of the gold inside its group, per question, for one global position.
    Every slot gets n_questions / group_size questions (+-1)."""
    order = np.random.default_rng([seed, position]).permutation(n_questions)
    slots = np.empty(n_questions, dtype=int)
    slots[order] = np.arange(n_questions) % group_size
    return slots


def split_with_gold(items: Sequence, gold_index: int, sizes: Sequence[int], slot: int) -> Tuple[List[list], int]:
    """Contiguous groups of `items`; the gold moves to `slot` inside its own group."""
    groups, start = [], 0
    for size in sizes:
        groups.append(list(items[start:start + size]))
        start += size
    gold_group = group_of(gold_index, sizes)
    group = groups[gold_group]
    gold = group.pop(gold_index - sum(sizes[:gold_group]))
    group.insert(slot, gold)
    return groups, gold_group


def content_key(*parts) -> str:
    """Stable short hash of JSON-serialisable parts."""
    blob = json.dumps(parts, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:24]


def contains_answer(text: str, answers: Sequence[str]) -> bool:
    """The metric's own rule (normalized substring); answers that normalize to "" never count."""
    norm = normalize_answer(text)
    return any(normalize_answer(a) and normalize_answer(a) in norm for a in answers)


def sample_ids(n_total: int, sample: str, limit=None, seed: int = SEED) -> List[int]:
    """'pilot': a seeded 200-question sample; 'all': every question. `limit` keeps the first N."""
    if sample == "pilot":
        rng = np.random.default_rng(seed)
        ids = sorted(int(i) for i in rng.choice(n_total, size=min(PILOT_SIZE, n_total), replace=False))
    elif sample == "all":
        ids = list(range(n_total))
    else:
        raise ValueError(f"Unknown sample: {sample}")
    return ids[:limit] if limit is not None else ids


def qa_map_items(root: Path, n_docs: int, positions: Sequence[int], question_ids: Sequence[int]) -> List[Dict]:
    """One item per (position, question, group), read from the authors' files."""
    task = f"qa{n_docs}"
    sizes = group_sizes(n_docs, QA_GROUP_SIZES[n_docs])
    items = []
    for position in positions:
        examples = read_jsonl(qa_path(root, position, n_docs))
        gold_group = group_of(position, sizes)
        slots = gold_slots(len(examples), position, sizes[gold_group])
        for idx in sorted(set(question_ids)):
            example = examples[idx]
            docs, rank = [], 0
            for ctx in example["ctxs"]:
                is_gold = bool(ctx.get("isgold"))
                docs.append({"title": ctx["title"], "text": ctx["text"], "isgold": is_gold, "rank": None if is_gold else rank})
                rank += 0 if is_gold else 1
            if not docs[position]["isgold"]:
                raise ValueError(f"{task} line {idx}: the gold document is not at index {position}")
            groups, _ = split_with_gold(docs, position, sizes, int(slots[idx]))
            for g, group in enumerate(groups):
                is_gold_group = g == gold_group
                items.append({
                    "task": task, "position": position, "idx": idx, "group": g,
                    "question": example["question"], "answers": example["answers"],
                    "docs": [{"title": d["title"], "text": d["text"]} for d in group],
                    "is_gold_group": is_gold_group,
                    "gold_slot": int(slots[idx]) if is_gold_group else None,
                    "mate_ranks": [d["rank"] for d in group if not d["isgold"]] if is_gold_group else None,
                    "n_answer_bearing": sum(
                        contains_answer(f"{d['title']} {d['text']}", example["answers"]) for d in group if not d["isgold"]
                    ),
                    "key": content_key(task, idx, [[d["title"], d["text"]] for d in group]),
                })
    return items


def kv_map_items(root: Path, positions: Sequence[int], question_ids: Sequence[int], n_keys: int = 300) -> List[Dict]:
    """One item per (position, example, group of 20 pairs). The gold pair is first moved to
    `position`, exactly as litm.data.kv_items does."""
    task = f"kv{n_keys}"
    examples = read_jsonl(kv_path(root, n_keys))
    sizes = group_sizes(n_keys, KV_GROUP_SIZE)
    items = []
    for position in positions:
        gold_group = group_of(position, sizes)
        slots = gold_slots(len(examples), position, sizes[gold_group])
        for idx in sorted(set(question_ids)):
            example = examples[idx]
            records = deepcopy(example["ordered_kv_records"])
            gold = records.pop(records.index([example["key"], example["value"]]))
            records.insert(position, gold)
            groups, _ = split_with_gold(records, position, sizes, int(slots[idx]))
            for g, group in enumerate(groups):
                is_gold_group = g == gold_group
                items.append({
                    "task": task, "position": position, "idx": idx, "group": g,
                    "question": example["key"], "query_key": example["key"], "answers": [example["value"]],
                    "docs": [list(pair) for pair in group],
                    "is_gold_group": is_gold_group,
                    "gold_slot": int(slots[idx]) if is_gold_group else None,
                    "mate_ranks": None,
                    "n_answer_bearing": 0,
                    "key": content_key(task, idx, group),
                })
    return items


In [ ]:
%%writefile litm/mapreduce/templates/kv_map.prompt
Extract the value corresponding to the specified key in the JSON object below. If the key is not in the JSON object, write NOT FOUND.

JSON data:
{formatted_kv_records}

Key: "{key}"
Reply in exactly this format:
Value: <the value, or NOT FOUND>
Evidence: <copy the "key": "value" pair from the JSON object>


In [ ]:
%%writefile litm/mapreduce/templates/qa_check.prompt
Question: {question}
Sentence: {evidence}
Proposed answer: {answer}
Does the sentence show that the proposed answer is correct? Answer Yes or No.


In [ ]:
%%writefile litm/mapreduce/templates/qa_judge.prompt
Question: {question}

Candidate answers, each with the sentence it came from:
{candidates}

Which candidate answers the question correctly? Reply in exactly this format:
Choice: <candidate number>


In [ ]:
%%writefile litm/mapreduce/templates/qa_map.prompt
Write a high-quality answer for the given question using only the provided search results (some of which might be irrelevant). If none of the search results contain the answer, write NOT FOUND.

{search_results}

Question: {question}
Reply in exactly this format:
Answer: <short answer, or NOT FOUND>
Evidence: <copy the sentence from the documents that contains the answer>


In [ ]:
%%writefile litm/mapreduce/verify.py
"""Keep only answers grounded in the group's own documents (spec §4.3, §9)."""
import re
from dataclasses import dataclass
from typing import Dict, Sequence

from rapidfuzz import fuzz

from litm.vendor.lost_in_the_middle.metrics import normalize_answer

MIN_EVIDENCE_WORDS = 5
FUZZY_THRESHOLD = 90  # rapidfuzz partial_ratio; frozen after the pilot (spec §8)
MIN_COVERAGE = 0.9    # the fuzzy match must cover 90% of the evidence, so real text + invented text fails

UUID = r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}"
UUID_RE = re.compile(UUID, re.I)
PAIR_RE = re.compile(r'"?\s*(' + UUID + r')\s*"?\s*:\s*"?\s*(' + UUID + r')', re.I)


@dataclass(frozen=True)
class Verdict:
    verified: bool
    how: str  # "exact" | "fuzzy" | why it was rejected


def _has_words(needle: str, haystack: str) -> bool:
    """Whole-word containment on normalized text ("one" is not inside "none")."""
    return f" {needle} " in f" {haystack} "


def _widen(text: str, start: int, end: int) -> str:
    """Grow [start, end) to whole words."""
    start = text.rfind(" ", 0, start) + 1
    stop = text.find(" ", end)
    return text[start:] if stop < 0 else text[start:stop]


def verify_qa(answer: str, evidence: str, docs: Sequence[Dict], fuzzy_threshold: float = FUZZY_THRESHOLD) -> Verdict:
    a, e = normalize_answer(answer), normalize_answer(evidence)
    if not a:
        return Verdict(False, "empty answer")
    if len(e.split()) < MIN_EVIDENCE_WORDS:
        return Verdict(False, "short evidence")
    if not _has_words(a, e):
        return Verdict(False, "answer not in evidence")
    texts = [normalize_answer(f"{d['title']} {d['text']}") for d in docs]
    if any(_has_words(e, t) for t in texts):
        return Verdict(True, "exact")
    for t in texts:
        match = fuzz.partial_ratio_alignment(e, t)
        if match is None:
            continue
        coverage = (match.src_end - match.src_start) / len(e)
        if (match.score >= fuzzy_threshold and coverage >= MIN_COVERAGE
                and _has_words(a, _widen(t, match.dest_start, match.dest_end))):
            return Verdict(True, "fuzzy")
    return Verdict(False, "evidence not in documents")


def verify_kv(value: str, evidence: str, pairs: Sequence[Sequence[str]], query_key: str) -> Verdict:
    value = value.strip().strip('"').lower()
    if not UUID_RE.fullmatch(value):
        return Verdict(False, "value not a UUID")
    match = PAIR_RE.search(evidence)
    if not match:
        return Verdict(False, "no key-value pair in evidence")
    key, evidence_value = match.group(1).lower(), match.group(2).lower()
    if key != query_key.lower():
        return Verdict(False, "evidence key is not the query key")
    if evidence_value != value:
        return Verdict(False, "value differs from evidence")
    if (key, value) not in {(k.lower(), v.lower()) for k, v in pairs}:
        return Verdict(False, "pair not in group")
    return Verdict(True, "exact")


In [ ]:
%%writefile litm/run.py
"""Run one model on a set of tasks with vLLM, saving every output as it goes.

Usage (one model per GPU):
    CUDA_VISIBLE_DEVICES=0 python -m litm.run --model qwen2.5-3b --tasks closedbook oracle qa20 --data <authors repo> --out results
    CUDA_VISIBLE_DEVICES=1 python -m litm.run --model qwen3-4b-2507 --tasks kv300 --kv-limit 200 ...

Rerunning the same command resumes: finished (task, position, idx) triples are skipped.
"""
import argparse
import hashlib
import json
import platform
import sys
import time
from pathlib import Path

from litm.data import all_items

MODELS = {
    "qwen2.5-3b": {"hf_id": "Qwen/Qwen2.5-3B-Instruct", "revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1"},
    "qwen3-4b-2507": {"hf_id": "Qwen/Qwen3-4B-Instruct-2507", "revision": "cdbee75f17c01a7cc42f958dc650907174af0554"},
}
MAX_NEW_TOKENS = 100


def key(record) -> tuple:
    return (record["task"], record["position"], record["idx"])


def load_done(path: Path) -> set:
    if not path.exists():
        return set()
    with open(path, encoding="utf-8") as f:
        return {key(json.loads(line)) for line in f if line.strip()}


def render(tokenizer, prompt: str) -> str:
    """The paper prompt as one user message in the model's own chat template, as shipped."""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )


def environment() -> dict:
    import torch
    import transformers
    import vllm

    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "vllm": vllm.__version__,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=sorted(MODELS), required=True)
    parser.add_argument("--tasks", nargs="+", required=True, help="closedbook oracle qa20 kv300")
    parser.add_argument("--data", type=Path, required=True, help="Path to the authors' repo checkout")
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--kv-limit", type=int, default=None, help="Use only the first N key-value examples")
    parser.add_argument("--qa-limit", type=int, default=None, help="Use only the first N questions (pilot/rerun)")
    parser.add_argument("--batch-size", type=int, default=512)
    parser.add_argument("--max-model-len", type=int, default=20608)  # longest prompt measured: 20,458 tokens (kv300) + 100 new tokens
    parser.add_argument("--enforce-eager", action="store_true")
    parser.add_argument("--tag", default="", help="Suffix for the output file, e.g. 'pilot' or 'rerun'")
    args = parser.parse_args(argv)

    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    from vllm.inputs import TokensPrompt

    model = MODELS[args.model]
    args.out.mkdir(parents=True, exist_ok=True)
    stem = f"{args.model}{'__' + args.tag if args.tag else ''}"
    out_path = args.out / f"{stem}.jsonl"
    meta_path = args.out / f"{stem}.meta.json"
    samples_path = args.out / f"{stem}.prompt_samples.jsonl"

    items = all_items(args.data, args.tasks, kv_limit=args.kv_limit)
    if args.qa_limit is not None:
        items = [x for x in items if x["task"].startswith("kv") or x["idx"] < args.qa_limit]
    done = load_done(out_path)
    pending = [x for x in items if key(x) not in done]
    print(f"{len(items)} items, {len(done)} already done, {len(pending)} to run", flush=True)
    if not pending:
        return

    # Render and tokenize everything first so over-long prompts fail before any GPU time is spent.
    tokenizer = AutoTokenizer.from_pretrained(model["hf_id"], revision=model["revision"])
    too_long = []
    seen_groups = set()
    rendered_all = [render(tokenizer, item["prompt"]) for item in pending]
    token_ids_all = tokenizer(rendered_all, add_special_tokens=False)["input_ids"]  # batched = parallel
    with open(samples_path, "a", encoding="utf-8") as samples:
        for item, rendered, token_ids in zip(pending, rendered_all, token_ids_all):
            item["token_ids"] = token_ids
            item["prompt_sha256"] = hashlib.sha256(rendered.encode("utf-8")).hexdigest()
            if len(item["token_ids"]) + MAX_NEW_TOKENS > args.max_model_len:
                too_long.append((key(item), len(item["token_ids"])))
            group = (item["task"], item["position"])
            if group not in seen_groups:  # one full rendered prompt per (task, position)
                seen_groups.add(group)
                samples.write(json.dumps({**{k: item[k] for k in ("task", "position", "idx")}, "rendered": rendered}) + "\n")
    if too_long:
        raise SystemExit(f"{len(too_long)} prompts exceed --max-model-len {args.max_model_len}, e.g. {too_long[:3]}")
    lengths = [len(x["token_ids"]) for x in pending]
    print(f"prompt tokens: max {max(lengths)}, mean {sum(lengths) / len(lengths):.0f}, total {sum(lengths)}", flush=True)

    llm = LLM(
        model=model["hf_id"],
        revision=model["revision"],
        dtype="float16",  # T4 has no bfloat16
        attention_backend="TRITON_ATTN",  # FlashAttention needs sm80+; FlashInfer has crashed on T4
        max_model_len=args.max_model_len,
        gpu_memory_utilization=0.9,
        enforce_eager=args.enforce_eager,
        seed=0,
    )
    sampling = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)

    meta = {
        "model": args.model,
        **model,
        "tasks": args.tasks,
        "kv_limit": args.kv_limit,
        "qa_limit": args.qa_limit,
        "max_model_len": args.max_model_len,
        "enforce_eager": args.enforce_eager,
        "max_new_tokens": MAX_NEW_TOKENS,
        "environment": environment(),
    }
    meta_path.write_text(json.dumps(meta, indent=2))

    started, tokens_done = time.time(), 0
    with open(out_path, "a", encoding="utf-8") as out:
        for start in range(0, len(pending), args.batch_size):
            batch = pending[start : start + args.batch_size]
            results = llm.generate([TokensPrompt(prompt_token_ids=x["token_ids"]) for x in batch], sampling, use_tqdm=False)
            for item, result in zip(batch, results):
                completion = result.outputs[0]
                record = {k: item[k] for k in ("task", "position", "idx", "answers", "prompt_sha256")}
                record.update(
                    model=args.model,
                    n_prompt_tokens=len(item["token_ids"]),
                    output=completion.text,
                    n_output_tokens=len(completion.token_ids),
                    finish_reason=completion.finish_reason,
                )
                out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()
            tokens_done += sum(len(x["token_ids"]) for x in batch)
            elapsed = time.time() - started
            print(
                f"{start + len(batch)}/{len(pending)} done, {elapsed / 60:.1f} min, {tokens_done / elapsed:.0f} prompt tok/s",
                flush=True,
            )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile litm/scoring.py
"""Scoring rules copied from the authors' evaluate_*_responses.py scripts.

One deliberate difference: we strip leading whitespace before taking the first line.
The authors' vLLM script strips generations before scoring, but their evaluate script
splits first, so an answer starting with a newline would score 0.
"""
import re
from typing import List

from litm.vendor.lost_in_the_middle.metrics import best_subspan_em, normalize_answer


def score_qa(output: str, answers: List[str]) -> int:
    first_line = output.strip().split("\n")[0].strip()
    return int(best_subspan_em(prediction=first_line, ground_truths=answers))


def score_kv(output: str, value: str) -> int:
    return int(value.lower() in output.lower())


def score(record: dict) -> int:
    if record["task"].startswith("kv"):
        return score_kv(record["output"], record["answers"][0])
    return score_qa(record["output"], record["answers"])


def strip_answer_prefix(answer: str) -> str:
    return re.sub(r"^\s*answer\s*:\s*", "", answer, flags=re.I).strip()


def score_final(answer: str, abstained: bool, answers: List[str]) -> int:
    """Map-reduce final answers (spec §5): an abstention scores 0 before the metric runs, so a
    gold answer like "S" can never match the words NOT FOUND."""
    if abstained or not answer.strip():
        return 0
    return int(best_subspan_em(prediction=strip_answer_prefix(answer), ground_truths=answers))


def strict_em(answer: str, abstained: bool, answers: List[str]) -> int:
    """Secondary metric: the normalized answer equals a normalized gold answer."""
    prediction = normalize_answer(strip_answer_prefix(answer))
    if abstained or not prediction:
        return 0
    return int(any(prediction == normalize_answer(gold) for gold in answers))


def score_final_kv(answer: str, abstained: bool, value: str) -> int:
    return 0 if abstained else score_kv(answer, value)


In [ ]:
%%writefile litm/stats.py
"""Paired statistics for position contrasts (plan section 5).

All comparisons are paired: a[i] and b[i] are 0/1 scores for the same question at two
positions.
"""
from math import comb, sqrt
from statistics import NormalDist
from typing import Dict, List, Sequence

import numpy as np

N_BOOTSTRAP = 10_000
SEED = 0
# Pre-registered equivalence margin: a contrast counts as "no effect" only if its whole
# 95% CI lies within +/- 3 points. (Using the MDE here would call tiny samples "Flat",
# because the MDE widens as fast as the CI.)
FLAT_MARGIN = 0.03


def accuracy_ci(scores: Sequence[int], n_boot: int = N_BOOTSTRAP, seed: int = SEED) -> Dict:
    x = np.asarray(scores, dtype=float)
    rng = np.random.default_rng(seed)
    means = x[rng.integers(0, len(x), size=(n_boot, len(x)))].mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return {"acc": x.mean(), "low": low, "high": high, "n": len(x)}


def paired_bootstrap(a: Sequence[int], b: Sequence[int], n_boot: int = N_BOOTSTRAP, seed: int = SEED) -> Dict:
    """95% CI for mean(a) - mean(b), resampling questions."""
    diff = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    means = diff[rng.integers(0, len(diff), size=(n_boot, len(diff)))].mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return {"diff": diff.mean(), "low": low, "high": high}


def mcnemar_exact(a: Sequence[int], b: Sequence[int]) -> Dict:
    """Two-sided exact McNemar test on discordant pairs."""
    a, b = np.asarray(a), np.asarray(b)
    only_a = int(((a == 1) & (b == 0)).sum())
    only_b = int(((a == 0) & (b == 1)).sum())
    n = only_a + only_b
    if n == 0:
        p = 1.0
    else:
        tail = sum(comb(n, k) for k in range(min(only_a, only_b) + 1)) / 2**n
        p = min(1.0, 2 * tail)
    return {"only_a": only_a, "only_b": only_b, "discordance": n / len(a), "p": p}


def holm(pvalues: List[float], alpha: float = 0.05) -> List[bool]:
    """Holm step-down: which hypotheses are rejected."""
    order = sorted(range(len(pvalues)), key=lambda i: pvalues[i])
    rejected = [False] * len(pvalues)
    for rank, i in enumerate(order):
        if pvalues[i] > alpha / (len(pvalues) - rank):
            break
        rejected[i] = True
    return rejected


def mde(discordance: float, n: int, alpha: float = 0.025, power: float = 0.8) -> float:
    """Minimum detectable paired difference (Card et al. 2020 approximation).

    alpha defaults to 0.05 / 2, the stricter Holm threshold for two contrasts.
    """
    z = NormalDist().inv_cdf(1 - alpha / 2) + NormalDist().inv_cdf(power)
    return z * sqrt(discordance / n)


def contrast(a: Sequence[int], b: Sequence[int]) -> Dict:
    result = {**paired_bootstrap(a, b), **mcnemar_exact(a, b)}
    result["mde"] = mde(result["discordance"], len(a))
    return result


def verdict(first: Sequence[int], middle: Sequence[int], last: Sequence[int]) -> Dict:
    """Pre-registered shape verdict from the first-vs-middle and last-vs-middle contrasts."""
    first_vs_middle = contrast(first, middle)
    last_vs_middle = contrast(last, middle)
    first_sig, last_sig = holm([first_vs_middle["p"], last_vs_middle["p"]])
    # A significant contrast only counts toward the shape if the middle is the lower one.
    first_up = first_sig and first_vs_middle["diff"] > 0
    last_up = last_sig and last_vs_middle["diff"] > 0

    if first_up and last_up:
        shape = "U shape"
    elif first_up and not last_sig:
        shape = "Primacy only"
    elif last_up and not first_sig:
        shape = "Recency only"
    elif not first_sig and not last_sig:
        within = all(
            -FLAT_MARGIN <= c["low"] and c["high"] <= FLAT_MARGIN for c in (first_vs_middle, last_vs_middle)
        )
        shape = "Flat" if within else "Inconclusive"
    else:
        shape = "Other"  # e.g. the middle is significantly *better* than an end
    return {"shape": shape, "first_vs_middle": first_vs_middle, "last_vs_middle": last_vs_middle}


# ---------------------------------------------------------------------------
# Map-reduce comparison (spec §6). Inputs are [questions, positions] arrays of 0/1 scores,
# rows aligned on the same question IDs, so every resample carries both methods together.

NI_MARGIN = 0.02  # pre-registered non-inferiority margin: 2 points


def bootstrap_mean(x: Sequence[float], n_boot: int = N_BOOTSTRAP, seed: int = SEED) -> Dict:
    """Mean of a per-question vector, with its percentile 95% CI and the bootstrap draws."""
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    boot = x[rng.integers(0, len(x), size=(n_boot, len(x)))].mean(axis=1)
    low, high = np.percentile(boot, [2.5, 97.5])
    return {"mean": float(x.mean()), "low": float(low), "high": float(high), "boot": boot}


def one_sided_p(boot: np.ndarray, threshold: float) -> float:
    """Bootstrap p-value for H0: mean <= threshold."""
    return float((np.sum(boot <= threshold) + 1) / (len(boot) + 1))


def u_contrast(scores) -> np.ndarray:
    """Per question: mean(first, last position) - mean(the positions in between)."""
    s = np.asarray(scores, dtype=float)
    return s[:, [0, -1]].mean(axis=1) - s[:, 1:-1].mean(axis=1)


def non_inferiority(new, ref, margin: float = NI_MARGIN) -> Dict:
    """Test 1: average over positions per question, then require CI low > -margin."""
    diff = np.asarray(new, dtype=float).mean(axis=1) - np.asarray(ref, dtype=float).mean(axis=1)
    r = bootstrap_mean(diff)
    return {"diff": r["mean"], "low": r["low"], "high": r["high"], "margin": margin,
            "p": one_sided_p(r["boot"], -margin), "pass": r["low"] > -margin}


def delta_u(base, new) -> Dict:
    """Test 2: U(baseline) - U(new) per question; pass if its CI is above 0 (new is flatter)."""
    u_base, u_new = u_contrast(base), u_contrast(new)
    r = bootstrap_mean(u_base - u_new)
    return {"u_base": float(u_base.mean()), "u_new": float(u_new.mean()), "diff": r["mean"],
            "low": r["low"], "high": r["high"], "p": one_sided_p(r["boot"], 0.0), "pass": r["low"] > 0}


def recovered_share(base, new, middle: int, first: int = 0, n_boot: int = N_BOOTSTRAP, seed: int = SEED) -> Dict:
    """Test 3: (new_mid - base_mid) / (base_first - base_mid), unstable if the denominator's CI touches 0."""
    b, n = np.asarray(base, dtype=float), np.asarray(new, dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(b), size=(n_boot, len(b)))
    num = n[:, middle][idx].mean(axis=1) - b[:, middle][idx].mean(axis=1)
    den = b[:, first][idx].mean(axis=1) - b[:, middle][idx].mean(axis=1)
    point_den = b[:, first].mean() - b[:, middle].mean()
    point = (n[:, middle].mean() - b[:, middle].mean()) / point_den if point_den else float("nan")
    ok = den != 0
    low, high = np.percentile(num[ok] / den[ok], [2.5, 97.5]) if ok.any() else (float("nan"), float("nan"))
    den_low, den_high = np.percentile(den, [2.5, 97.5])
    return {"share": float(point), "low": float(low), "high": float(high), "unstable": bool(den_low <= 0 <= den_high)}


def range_minus_null(scores, n_perm: int = N_BOOTSTRAP, seed: int = SEED, chunk: int = 250) -> Dict:
    """Best-minus-worst position accuracy, and its mean when position labels are shuffled
    within each question (the "no position effect" value). Descriptive only."""
    s = np.asarray(scores, dtype=float)
    acc = s.mean(axis=0)
    rng = np.random.default_rng(seed)
    null = []
    for start in range(0, n_perm, chunk):
        k = min(chunk, n_perm - start)
        order = np.argsort(rng.random((k, *s.shape)), axis=2)
        means = np.take_along_axis(np.broadcast_to(s, (k, *s.shape)), order, axis=2).mean(axis=1)
        null.append(means.max(axis=1) - means.min(axis=1))
    null_mean = float(np.concatenate(null).mean())
    observed = float(acc.max() - acc.min())
    return {"range": observed, "null_mean": null_mean, "excess": observed - null_mean}


def auroc(scores: Sequence[float], labels: Sequence[int]) -> float:
    """Area under the ROC curve (ties count half); nan if either class is missing."""
    s, y = np.asarray(scores, dtype=float), np.asarray(labels, dtype=bool)
    n_pos, n_neg = int(y.sum()), int((~y).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    _, inverse, counts = np.unique(s, return_inverse=True, return_counts=True)
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s))
    ranks[order] = np.arange(1, len(s) + 1)
    ranks = (np.bincount(inverse, weights=ranks) / counts)[inverse]
    return float((ranks[y].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/LICENSE
MIT License

Copyright (c) 2023 Nelson Liu

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/metrics.py
#!/usr/bin/env python3
import string
from typing import List

import regex


def normalize_answer(s: str) -> str:
    """Normalization from the SQuAD evaluation script.

    See https://worksheets.codalab.org/rest/bundles/0x6b567e1cf2e041ec80d7098f031c5c9e/contents/blob/
    """

    def remove_articles(text):
        return regex.sub(r"\b(a|an|the)\b", " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def best_subspan_em(prediction: str, ground_truths: List[str]) -> float:
    normalized_prediction = normalize_answer(prediction)

    for ground_truth in ground_truths:
        normalized_ground_truth = normalize_answer(ground_truth)
        if normalized_ground_truth.lower() in normalized_prediction.lower():
            return 1.0
    return 0.0


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompting.py
#!/usr/bin/env python3
import pathlib
from copy import deepcopy
from typing import List, Optional, Tuple, Type, TypeVar

from pydantic.dataclasses import dataclass

PROMPTS_ROOT = (pathlib.Path(__file__).parent / "prompts").resolve()

T = TypeVar("T")


@dataclass(frozen=True)
class Document:
    title: str
    text: str
    id: Optional[str] = None
    score: Optional[float] = None
    hasanswer: Optional[bool] = None
    isgold: Optional[bool] = None
    original_retrieval_index: Optional[int] = None

    @classmethod
    def from_dict(cls: Type[T], data: dict) -> T:
        data = deepcopy(data)
        if not data:
            raise ValueError("Must provide data for creation of Document from dict.")
        id = data.pop("id", None)
        score = data.pop("score", None)
        # Convert score to float if it's provided.
        if score is not None:
            score = float(score)
        return cls(**dict(data, id=id, score=score))


def get_qa_prompt(
    question: str, documents: List[Document], mention_random_ordering: bool, query_aware_contextualization: bool
):
    if not question:
        raise ValueError(f"Provided `question` must be truthy, got: {question}")
    if not documents:
        raise ValueError(f"Provided `documents` must be truthy, got: {documents}")

    if mention_random_ordering and query_aware_contextualization:
        raise ValueError("Mentioning random ordering cannot be currently used with query aware contextualization")

    if mention_random_ordering:
        prompt_filename = "qa_ordered_randomly.prompt"
    elif query_aware_contextualization:
        prompt_filename = "qa_with_query_aware_contextualization.prompt"
    else:
        prompt_filename = "qa.prompt"

    with open(PROMPTS_ROOT / prompt_filename) as f:
        prompt_template = f.read().rstrip("\n")

    # Format the documents into strings
    formatted_documents = []
    for document_index, document in enumerate(documents):
        formatted_documents.append(f"Document [{document_index+1}](Title: {document.title}) {document.text}")
    return prompt_template.format(question=question, search_results="\n".join(formatted_documents))


def get_closedbook_qa_prompt(question: str):
    if not question:
        raise ValueError(f"Provided `question` must be truthy, got: {question}")
    with open(PROMPTS_ROOT / "closedbook_qa.prompt") as f:
        prompt_template = f.read().rstrip("\n")

    return prompt_template.format(question=question)


def get_kv_retrieval_prompt(
    data: List[Tuple[str, str]],
    key: str,
    query_aware_contextualization: bool = False,
):
    if not data:
        raise ValueError(f"Provided `data` must be truthy, got: {data}")
    if not key:
        raise ValueError(f"Provided `key` must be truthy, got: {key}")
    if key not in [x[0] for x in data]:
        raise ValueError(f"Did not find provided `key` {key} in data {data}")
    if len(data) != len(set([x[0] for x in data])):
        raise ValueError(f"`data` has duplicate keys: {data}")
    if len(data) < 2:
        raise ValueError(f"Must have at least 2 items in data: {data}")

    if query_aware_contextualization:
        with open(PROMPTS_ROOT / "kv_retrieval_with_query_aware_contextualization.prompt") as f:
            prompt_template = f.read().rstrip("\n")
    else:
        with open(PROMPTS_ROOT / "kv_retrieval.prompt") as f:
            prompt_template = f.read().rstrip("\n")

    # Format the KV data into a string
    formatted_kv_records = ""
    for index, record in enumerate(data):
        start_character = "{" if index == 0 else " "
        data_string = f'"{record[0]}": "{record[1]}"'
        end_character = ",\n" if index != len(data) - 1 else "}"
        formatted_kv_records += start_character + data_string + end_character

    return prompt_template.format(formatted_kv_records=formatted_kv_records, key=key)


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/closedbook_qa.prompt
Question: {question}
Answer:


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/kv_retrieval.prompt
Extract the value corresponding to the specified key in the JSON object below.

JSON data:
{formatted_kv_records}

Key: "{key}"
Corresponding value:


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/kv_retrieval_with_query_aware_contextualization.prompt
Extract the value corresponding to the specified key in the JSON object below.

Key: "{key}"

JSON data:
{formatted_kv_records}

Key: "{key}"
Corresponding value:


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/qa.prompt
Write a high-quality answer for the given question using only the provided search results (some of which might be irrelevant).

{search_results}

Question: {question}
Answer:

In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/qa_ordered_randomly.prompt
Write a high-quality answer for the given question using only the provided search results (some of which might be irrelevant). The search results are ordered randomly.

{search_results}

Question: {question}
Answer:

In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/qa_with_query_aware_contextualization.prompt
Write a high-quality answer for the given question using only the provided search results (some of which might be irrelevant).

Question: {question}

{search_results}

Question: {question}
Answer:


In [ ]:
import glob, os, shutil, subprocess, sys, time
WORK = "/kaggle/working"
VENV = "/tmp/vllm-env"
AUTHORS = "/tmp/lost-in-the-middle"
AUTHORS_COMMIT = "29b8a6d042ce29abccee3db1a73171a107d7e6af"
os.chdir(WORK)

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

# Isolated environment for vLLM 0.18.1 (it pins its own torch 2.10.0); this kernel never imports torch.
if not os.path.exists(f"{VENV}/bin/python"):
    sh("pip install -q uv")
    sh(f"uv venv -q {VENV} --python 3.12")
    sh(f"uv pip install -q --python {VENV}/bin/python vllm==0.18.1 pydantic regex xopen numpy matplotlib rapidfuzz")
PY = f"{VENV}/bin/python"

# The authors' data and code at a pinned commit.
if not os.path.exists(AUTHORS):
    sh(f"git init -q {AUTHORS} && cd {AUTHORS} && git fetch -q --depth 1 "
       f"https://github.com/nelson-liu/lost-in-the-middle.git {AUTHORS_COMMIT} && git checkout -q FETCH_HEAD")

# Resume: copy results from any attached earlier version of this notebook.
os.makedirs("results", exist_ok=True)
for path in glob.glob("/kaggle/input/**/results/*.jsonl", recursive=True):
    target = os.path.join("results", os.path.basename(path))
    if not os.path.exists(target):
        shutil.copy(path, target)
        print("resumed", target)
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")


## Run the stage (one model per GPU)

In [ ]:
STAGES = {
    # stage: (extra args, output tag, tasks)
    "smoke": (["--qa-limit", "2", "--kv-limit", "1"], "smoke", ["closedbook", "oracle", "qa20", "kv300"]),
    "pilot": (["--qa-limit", "20", "--kv-limit", "4"], "pilot", ["closedbook", "oracle", "qa20", "kv300"]),
    "qa":    ([], "", ["closedbook", "oracle", "qa20"]),
    "qa30":  ([], "", ["qa30"]),
    "kv":    (["--kv-limit", str(KV_LIMIT)], "", ["kv300"]),
    "rerun": (["--qa-limit", "300", "--kv-limit", "50"], "rerun", ["qa20", "kv300"]),
}
MR_STAGES = ["mr_map", "mr_check", "mr_judge", "mr_control"]  # GPU; mr_reduce and mr_analyze run on CPU
MODELS = ["qwen2.5-3b", "qwen3-4b-2507"]  # model i runs on GPU i
MR_ARGS = ["--task", MR_TASK, "--questions", MR_SAMPLE, "--data", AUTHORS, "--out", "results"]
if MR_TASK.startswith("kv"):
    MR_ARGS += ["--kv-limit", str(KV_LIMIT)]  # the same examples as the baseline
if MR_LIMIT is not None:
    MR_ARGS += ["--limit", str(MR_LIMIT)]
if MR_CONTROL_SUBSET is not None:
    MR_ARGS += ["--control-subset", str(MR_CONTROL_SUBSET)]  # reduce then uses the control outputs that exist

def launch(commands):
    # Run one (model, command, log file) per GPU in parallel; report progress every 5 minutes.
    procs = []
    for gpu, (model, cmd, log_name) in enumerate(commands):
        if ENFORCE_EAGER:
            cmd = [*cmd, "--enforce-eager"]
        log = open(log_name, "a")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONPATH": WORK}
        procs.append((model, log_name, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)))
        print(f"GPU {gpu}: {model} -> {log_name}")
    started = time.time()
    while any(p.poll() is None for _, _, p in procs):
        time.sleep(300)
        print(f"--- {(time.time() - started) / 60:.0f} min")
        for model, log_name, _ in procs:
            lines = open(log_name).read().splitlines()
            print(model, "|", lines[-1] if lines else "")
    for model, log_name, p in procs:
        print(f"{model}: exit code {p.returncode}")
        if p.returncode != 0:
            print("".join(open(log_name).readlines()[-40:]))

if STAGE in STAGES:
    extra, tag, tasks = STAGES[STAGE]
    commands = []
    for model in MODELS:
        cmd = [PY, "-m", "litm.run", "--model", model, "--tasks", *tasks, "--data", AUTHORS,
               "--out", "results", "--max-model-len", str(MAX_MODEL_LEN), *extra]
        if tag:
            cmd += ["--tag", tag]
        commands.append((model, cmd, f"results/{model}{'__' + tag if tag else ''}.{STAGE}.log"))
    launch(commands)
elif STAGE in MR_STAGES:
    stage = STAGE[len("mr_"):]
    launch([(model, [PY, "-m", "litm.mapreduce.run", "--model", model, "--stage", stage, *MR_ARGS],
             f"results/{model}__mr-{stage}.{MR_TASK}-{MR_SAMPLE}.log") for model in MODELS])
elif STAGE == "mr_reduce":
    for model in MODELS:
        sh(" ".join([f"PYTHONPATH={WORK}", PY, "-m", "litm.mapreduce.run", "--model", model, "--stage", "reduce", *MR_ARGS]))


In [ ]:
# Smoke/pilot check: read the outputs by hand before any full run.
import json
if STAGE == "mr_map":  # spec §8: read 20 raw map outputs per model before the full run
    for model in MODELS:
        print("=" * 30, model)
        with open(f"results/{model}__mr-map.jsonl") as f:
            for line, _ in zip(f, range(20)):
                print(repr(json.loads(line)["text"][:300]))
if STAGE in ("smoke", "pilot"):
    for model in MODELS:
        path = f"results/{model}__{STAGE}.jsonl"
        if not os.path.exists(path):
            continue
        print("=" * 30, model)
        for line in open(path):
            r = json.loads(line)
            if STAGE == "smoke" or r["idx"] < 2:
                print(f"{r['task']}@{r['position']} #{r['idx']} ({r['n_prompt_tokens']} tok, {r['finish_reason']}): "
                      f"{r['output'][:120]!r}  gold={r['answers'][:2]}")


## Score and analyse

In [ ]:
if STAGE == "mr_analyze":
    sh(" ".join([f"PYTHONPATH={WORK}", PY, "-m", "litm.mapreduce.analyze", "results",
                 "--task", MR_TASK, "--sample", MR_SAMPLE, "--out", "analysis"]))
elif not STAGE.startswith("mr_"):
    tag_args = ["--tag", STAGES[STAGE][1]] if STAGE in STAGES and STAGES[STAGE][1] else []
    sh(" ".join([f"PYTHONPATH={WORK}", PY, "-m", "litm.analyze", "results", "--out", "analysis", *tag_args]))
